# ARG Demo
## Getting the data
Before any of the fun starts we need to actually get the data, which wasn't copied to the server when we cloned the repo. To get the data do the following in the terminal:
1. ```conda install conda-forge::git-lfs```
2. ```git lfs install```
3. ```git lfs pull```
## Looking at the Data
Now that we have the data we can start working with it, but before that, we should look at the data we will input into SINGER.
To do so we first need to install bcftools which, can be done by running: ```conda install -c bioconda bcftools```

Once that is done, we can run: ```bcftools view ./data/abo.vcf.gz | less -S```
This lets us get a glimpse of the data in a readable format.

> **Note**: Since we are working with human data we avoid a lot of the hastle of imputing and phasing the data.

## On to the fun part
The tool we will be using is called SINGER, which was developed by a former member of my lab. It is an MCMC based method that works by slowly adding one branch on a time to the tree and then iterating through the tree space, similarly to what we did last week with Enzo. Since this method is Bayesian based, our output is a distribution of trees from which we can gather our statistics.

### Envrionment setup
We need a place for our output, lets call it singer_out
`mkdir singer_out`

### Unzipping the data
1. `cd data`
2. `gunzip -c abo.vcf.gz > abo.vcf`
3. `cd ..`

In [ ]:
import subprocess

# All of our parameters
mutation_rate = 2.5e-8
# Recombination rates change across the genome, this accounts for this variability
recomb_map = './data/maps.pat.tsv'
Ne = 31e2
start = 1326e5
end = 1338e5
vcf = 'data/abo'

singer_command = f"./singer-0.1.9-beta-linux-x86_64/singer_master -Ne {Ne} -m {mutation_rate} -recomb_map {recomb_map} -start {start} -end {end}  -vcf {vcf} -output singer_out/out.chr9"

subprocess.run(singer_command, shell=True)

### Converting to tskit
The output from Singer isn't quite ready for analysis. We need to convert it to a format we can work with using the tskit library

In [ ]:
convert_command = """./singer-0.1.9-beta-linux-x86_64/convert_to_tskit \
            -input ./singer_out/out.chr9 \
            -output ./singer_out/chr9.tsk \
            -start 0 \
            -end 100 \
            -step 10"""

subprocess.run(convert_command, shell=True)

### Looking at a tree
Usually this isn't that practical but is still cool.

In [ ]:
%conda install -c conda-forge tskit
import tskit
from IPython.display import SVG, display

ts = tskit.load('./singer_out/out.chr9_5.trees')
tree_svg = ts.first().draw_svg(size=(400, 300))
display(SVG(tree_svg))

### Checking convergence
To make sure our trees our accurate we need to look at the convergence of the mcmc.

In [ ]:
# Getting convergence statistics
get_traces = """ python ./singer-0.1.9-beta-linux-x86_64/compute_trace.py
            -prefix ./singer_out/chr9.tsk
            -m 2.5e-8
            -start_index 0
            -end_index 10
            -step 1
            -output_filename ./singer_out/traces.tsv"""
subprocess.run(get_traces, shell=True)

In [ ]:
# Plotting convergence
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'./singer_out/traces.tsv', sep='\t')

fig1, ax1 = plt.subplots()
fig2, ax2 = plt.subplots()

ax1.plot('indices', 'div_traces', data=df)
ax1.axhline(y=df['div_traces'].median(), color="black", ls='--')
plt.show()
ax2.plot('indices', 'inc_traces', data=df)
ax2.axhline(y=df['inc_traces'].median(), color="black", ls='--')
plt.show()

## Analyses
For this tutorial we want to show that as a result of balancing selection on the ABO gene, the TMRCA for the gene is really old and lines up with when humans divereged from chimpanzees (around 6 million years ago).

In [ ]:
# Function to calculate TMRCA for a given tree and range of the sequence
import numpy as np

def calculate_windowed_tmrca(ts, window_size):
    grid = np.arange(0, ts.sequence_length, window_size)
    grid = np.append(grid, ts.sequence_length)
    window_index = 0
    m = len(grid) - 1
    arg_tmrca = np.zeros(m)
    tree = ts.first()
    while tree.interval[1] != ts.sequence_length and window_index <= m:
        y = min(tree.interval[1], grid[window_index + 1])
        x = max(tree.interval[0], grid[window_index])
        if len(tree.roots) == 1:
            arg_tmrca[window_index] += tree.time(tree.root)*(y-x)/window_size
        if tree.interval[1] > grid[window_index + 1]:
            window_index += 1
        else:
            tree.next()
    return arg_tmrca

In [ ]:
ceu_chr9_ts_list = []
# Look at only the last half of the trees
for i in range(50, 100, 10):
    ceu_chr9_ts = tskit.load(f"./singer_out/chr9_{i}.trees")
    ceu_chr9_ts_list.append(ceu_chr9_ts)

ceu_tmrca_list = []
for ceu_chr9_ts in ceu_chr9_ts_list: 
    y = calculate_windowed_tmrca(ceu_chr9_ts, 1e3)
    ceu_tmrca_list.append(y)

ceu_mean_tmrca = np.mean(ceu_tmrca_list, axis=0)
ceu_median_tmrca = np.median(ceu_mean_tmrca, axis=0)

plt.figure(figsize = [12, 3])
plt.plot([0.001*i for i in range(len(ceu_mean_tmrca))], ceu_mean_tmrca*28/1e6, color='purple', label="Inferred TMRCA")
plt.xlim((132.9, 133.5));
plt.fill_between([132.7+0.533, 132.7+0.576], y1=0, y2=9e6, color='gray', alpha=0.5, label='ABO');
plt.ylim((0, 10));
plt.axhline(y=6, color='black', linestyle='-');
plt.axhline(y=np.mean(ceu_median_tmrca)*28/1e6, color='brown', linestyle='--', label='Chr9 median TMRCA');
plt.legend(fontsize = 10);
plt.show()